In [1]:
import joblib
import os
import pandas as pd
from IPython.display import display

## Load Pre-trained Models

Load the Naive Bayes models and vectorizers trained in the individual notebooks:
- Spam detection model (trained on Enron dataset)
- Phishing detection model (trained on Phishing Email dataset)

In [2]:
models_dir = '../models'

# Load spam detection model and vectorizer
spam_model = joblib.load(os.path.join(models_dir, 'nb_model.joblib'))
spam_vectorizer = joblib.load(os.path.join(models_dir, 'nb_vectorizer.joblib'))

# Load phishing detection model and vectorizer
phishing_model = joblib.load(os.path.join(models_dir, 'phishing_nb_model.joblib'))
phishing_vectorizer = joblib.load(os.path.join(models_dir, 'phishing_nb_vectorizer.joblib'))

print("Spam detection model loaded")
print("Phishing detection model loaded")

Spam detection model loaded
Phishing detection model loaded


## Define Pipeline Classification Function

In [3]:
def classify_email(text, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer):
    """
    Classify an email through the spam → phishing pipeline.
    
    Returns:
        dict with classification results including:
        - final_label: 'HAM', 'SPAM', or 'PHISHING'
        - spam_prediction: 0 (ham) or 1 (spam)
        - spam_confidence: confidence score for spam prediction
        - phishing_prediction: 0 (safe) or 1 (phishing), None if ham
        - phishing_confidence: confidence score for phishing, None if ham
    """
    result = {
        'text': text,
        'spam_prediction': None,
        'spam_confidence': None,
        'spam_probabilities': None,
        'phishing_prediction': None,
        'phishing_confidence': None,
        'phishing_probabilities': None,
        'final_label': None
    }
    
    # Stage 1: Spam Detection
    text_vec_spam = spam_vectorizer.transform([text])
    spam_pred = spam_model.predict(text_vec_spam)[0]
    spam_probs = spam_model.predict_proba(text_vec_spam)[0]
    
    result['spam_prediction'] = spam_pred
    result['spam_confidence'] = spam_probs[spam_pred]
    result['spam_probabilities'] = {'ham': spam_probs[0], 'spam': spam_probs[1]}
    
    # If ham (not spam), we're done
    if spam_pred == 0:
        result['final_label'] = 'HAM'
        return result
    
    # Stage 2: Phishing Detection (only for spam emails)
    text_vec_phishing = phishing_vectorizer.transform([text])
    phishing_pred = phishing_model.predict(text_vec_phishing)[0]
    phishing_probs = phishing_model.predict_proba(text_vec_phishing)[0]
    
    result['phishing_prediction'] = phishing_pred
    result['phishing_confidence'] = phishing_probs[phishing_pred]
    result['phishing_probabilities'] = {'safe': phishing_probs[0], 'phishing': phishing_probs[1]}
    
    # Final classification
    if phishing_pred == 1:
        result['final_label'] = 'PHISHING'
    else:
        result['final_label'] = 'SPAM'
    
    return result

## Pretty Print Results Function

In [4]:
def display_result(result, example_num=None):
    """
    Display classification results in a user-friendly format.
    """
    # Define colors and icons for each label
    label_styles = {
        'HAM': {'color': '#28a745', 'icon': '✅', 'bg': '#d4edda'},
        'SPAM': {'color': '#ffc107', 'icon': '⚠️', 'bg': '#fff3cd'},
        'PHISHING': {'color': '#dc3545', 'icon': '🚨', 'bg': '#f8d7da'}
    }
    
    style = label_styles[result['final_label']]
    
    # Header
    header = f"Example {example_num}" if example_num else "Classification Result"
    print(f"\n{'='*70}")
    print(f"{header}")
    print(f"{'='*70}")
    
    # Email text (truncated if too long)
    text = result['text']
    display_text = text[:150] + "..." if len(text) > 150 else text
    print(f"\n📧 Email: {display_text}")
    
    # Stage 1 results
    print(f"\n--- Stage 1: Spam Detection ---")
    spam_label = "SPAM" if result['spam_prediction'] == 1 else "HAM"
    print(f"Prediction: {spam_label}")
    print(f"Confidence: {result['spam_confidence']:.2%}")
    print(f"Probabilities: Ham={result['spam_probabilities']['ham']:.2%}, Spam={result['spam_probabilities']['spam']:.2%}")
    
    # Stage 2 results (only if spam)
    if result['phishing_prediction'] is not None:
        print(f"\n--- Stage 2: Phishing Detection ---")
        phishing_label = "PHISHING" if result['phishing_prediction'] == 1 else "SAFE (Regular Spam)"
        print(f"Prediction: {phishing_label}")
        print(f"Confidence: {result['phishing_confidence']:.2%}")
        print(f"Probabilities: Safe={result['phishing_probabilities']['safe']:.2%}, Phishing={result['phishing_probabilities']['phishing']:.2%}")
    else:
        print(f"\n--- Stage 2: Phishing Detection ---")
        print("Skipped (email is not spam)")
    
    # Final result
    print(f"\n{'='*70}")
    print(f"{style['icon']} FINAL CLASSIFICATION: {result['final_label']} {style['icon']}")
    print(f"{'='*70}")

## Test the Pipeline

Let's test the pipeline with various types of emails:
1. Legitimate business emails (should be classified as HAM)
2. Regular spam emails (should be classified as SPAM)
3. Phishing emails (should be classified as PHISHING)

In [5]:
# Test examples covering all three categories
test_emails = [
    # Legitimate emails (expected: HAM)
    {
        'text': "Hi Team, Please find attached the quarterly report. Let me know if you have any questions. Best regards, John",
        'expected': 'HAM',
        'description': 'Legitimate business email'
    },
    {
        'text': "Meeting reminder: Project sync tomorrow at 2pm in Conference Room B. Agenda has been shared.",
        'expected': 'HAM',
        'description': 'Meeting reminder'
    },
    {
        'text': "Can you review the contract and send me your feedback by Friday? Thanks!",
        'expected': 'HAM',
        'description': 'Work request'
    },
    
    # Regular spam emails (expected: SPAM)
    {
        'text': "AMAZING DEALS! Buy one get one FREE! Limited time offer on all products! Shop now and save big!",
        'expected': 'SPAM',
        'description': 'Marketing spam'
    },
    {
        'text': "You have been selected for a special discount! Act now to receive 50% off your next purchase!",
        'expected': 'SPAM',
        'description': 'Promotional spam'
    },
    
    # Phishing emails (expected: PHISHING)
    {
        'text': "URGENT: Your account has been compromised! Click here immediately to verify your identity and secure your account before it's too late.",
        'expected': 'PHISHING',
        'description': 'Account compromise phishing'
    },
    {
        'text': "Your PayPal account has been limited. Please update your information immediately to avoid suspension. Click here to verify your account.",
        'expected': 'PHISHING',
        'description': 'PayPal phishing'
    },
    {
        'text': "Dear Customer, We detected unusual activity on your bank account. Please confirm your identity by providing your SSN and password immediately.",
        'expected': 'PHISHING',
        'description': 'Banking phishing'
    },
    {
        'text': "Congratulations! You have won $1,000,000 in the lottery! Click the link below and enter your credit card information to claim your prize now!",
        'expected': 'PHISHING',
        'description': 'Lottery scam phishing'
    }
]

# Run classification for each test email
results = []
for i, email in enumerate(test_emails, 1):
    result = classify_email(
        email['text'],
        spam_model, spam_vectorizer,
        phishing_model, phishing_vectorizer
    )
    result['expected'] = email['expected']
    result['description'] = email['description']
    results.append(result)
    display_result(result, example_num=i)


Example 1

📧 Email: Hi Team, Please find attached the quarterly report. Let me know if you have any questions. Best regards, John

--- Stage 1: Spam Detection ---
Prediction: HAM
Confidence: 97.42%
Probabilities: Ham=97.42%, Spam=2.58%

--- Stage 2: Phishing Detection ---
Skipped (email is not spam)

✅ FINAL CLASSIFICATION: HAM ✅

Example 2

📧 Email: Meeting reminder: Project sync tomorrow at 2pm in Conference Room B. Agenda has been shared.

--- Stage 1: Spam Detection ---
Prediction: HAM
Confidence: 98.63%
Probabilities: Ham=98.63%, Spam=1.37%

--- Stage 2: Phishing Detection ---
Skipped (email is not spam)

✅ FINAL CLASSIFICATION: HAM ✅

Example 3

📧 Email: Can you review the contract and send me your feedback by Friday? Thanks!

--- Stage 1: Spam Detection ---
Prediction: HAM
Confidence: 97.47%
Probabilities: Ham=97.47%, Spam=2.53%

--- Stage 2: Phishing Detection ---
Skipped (email is not spam)

✅ FINAL CLASSIFICATION: HAM ✅

Example 4

📧 Email: AMAZING DEALS! Buy one get one FRE

## Pipeline Accuracy Summary

In [6]:
# Create summary table
summary_data = []
for i, r in enumerate(results, 1):
    match = '✅' if r['final_label'] == r['expected'] else '❌'
    summary_data.append({
        'Example': i,
        'Description': r['description'],
        'Expected': r['expected'],
        'Predicted': r['final_label'],
        'Match': match
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("PIPELINE CLASSIFICATION SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))

# Calculate accuracy
correct = sum(1 for r in results if r['final_label'] == r['expected'])
total = len(results)
accuracy = correct / total

print(f"\n{'='*70}")
print(f"Pipeline Accuracy on Test Examples: {correct}/{total} ({accuracy:.1%})")
print(f"{'='*70}")


PIPELINE CLASSIFICATION SUMMARY
 Example                 Description Expected Predicted Match
       1   Legitimate business email      HAM       HAM     ✅
       2            Meeting reminder      HAM       HAM     ✅
       3                Work request      HAM       HAM     ✅
       4              Marketing spam     SPAM  PHISHING     ❌
       5            Promotional spam     SPAM  PHISHING     ❌
       6 Account compromise phishing PHISHING  PHISHING     ✅
       7             PayPal phishing PHISHING  PHISHING     ✅
       8            Banking phishing PHISHING  PHISHING     ✅
       9       Lottery scam phishing PHISHING  PHISHING     ✅

Pipeline Accuracy on Test Examples: 7/9 (77.8%)


## Interactive Email Classifier

Try classifying your own emails! Modify the `user_email` variable below.

In [7]:
# Enter your own email text to classify
user_email = """
Dear valued customer,

We have detected suspicious activity on your Amazon account. 
Your account will be suspended unless you verify your information within 24 hours.

Click here to verify: http://amaz0n-security.fake-domain.com/verify

Please provide your login credentials and payment information to restore access.

Regards,
Amazon Security Team
"""

# Classify the email
result = classify_email(
    user_email.strip(),
    spam_model, spam_vectorizer,
    phishing_model, phishing_vectorizer
)

display_result(result)


Classification Result

📧 Email: Dear valued customer,

We have detected suspicious activity on your Amazon account. 
Your account will be suspended unless you verify your information...

--- Stage 1: Spam Detection ---
Prediction: SPAM
Confidence: 85.73%
Probabilities: Ham=14.27%, Spam=85.73%

--- Stage 2: Phishing Detection ---
Prediction: PHISHING
Confidence: 95.64%
Probabilities: Safe=4.36%, Phishing=95.64%

🚨 FINAL CLASSIFICATION: PHISHING 🚨


## Batch Classification Function

For processing multiple emails at once.

In [8]:
def classify_batch(emails, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer):
    """
    Classify a batch of emails through the pipeline.
    
    Parameters:
        emails: list of email text strings
        
    Returns:
        DataFrame with classification results
    """
    results = []
    
    for email in emails:
        result = classify_email(
            email,
            spam_model, spam_vectorizer,
            phishing_model, phishing_vectorizer
        )
        results.append({
            'email_preview': email[:80] + '...' if len(email) > 80 else email,
            'spam_detected': result['spam_prediction'] == 1,
            'spam_confidence': result['spam_confidence'],
            'phishing_detected': result['phishing_prediction'] == 1 if result['phishing_prediction'] is not None else None,
            'phishing_confidence': result['phishing_confidence'],
            'final_label': result['final_label']
        })
    
    return pd.DataFrame(results)

# Example batch classification
batch_emails = [
    "Team meeting at 3pm today in the main conference room.",
    "FREE GIFT! You've been selected! Click now to claim!",
    "Your password expires in 24 hours. Click here to reset immediately or lose access.",
    "Quarterly results are attached. Please review before tomorrow's call."
]

batch_results = classify_batch(
    batch_emails,
    spam_model, spam_vectorizer,
    phishing_model, phishing_vectorizer
)

print("Batch Classification Results:")
print("="*100)
display(batch_results)

Batch Classification Results:


,email_preview,spam_detected,spam_confidence,phishing_detected,phishing_confidence,final_label
0,Team meeting at 3pm today in the main conferen...,False,0.954395,None,NaN,HAM
1,FREE GIFT! You've been selected! Click now to ...,True,0.970858,True,0.915562,PHISHING
2,Your password expires in 24 hours. Click here ...,True,0.845099,True,0.846975,PHISHING
3,Quarterly results are attached. Please review ...,False,0.933910,None,NaN,HAM
